# L12 · GRPO와 verifiable reward

## Goal

- group-relative advantage를 계산한다
- critic 제거 trade-off를 설명한다
- zero-variance와 length bias를 진단한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L12:toy:42").hexdigest()
print(f"lesson=L12 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L12 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:c261f138b02c7389fc58467b8b2b0502ad850b154b2d0a4ddd24631066a0431d data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: LLM policy + verifier → **GRPO·RLVR** → DAPO

$$\hat A_i=\frac{r_i-\operatorname{mean}(r_{1:G})}{\operatorname{std}(r_{1:G})+\epsilon}$$

GRPO는 같은 prompt에서 G개 completion을 뽑고 group reward 평균을 baseline으로 사용해 별도 critic을 없앱니다. RLVR은 정답·형식처럼 검증 가능한 reward를 사용해 reward model의 모호함을 줄입니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** 한 group의 reward가 모두 2라면 normalized advantage는 NaN일까요, 0일까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>안전한 구현은 0으로 만들고 그 group을 informative하지 않다고 표시합니다.</details>

In [2]:
from rl_study.algorithms.grpo import group_relative_advantages, rloo_advantages
group_rewards = torch.tensor([[0.0, 0.0, 1.0, 1.0], [2.0, 2.0, 2.0, 2.0]])
grpo_adv = group_relative_advantages(group_rewards)
rloo_adv = rloo_advantages(group_rewards)
print({"grpo_advantages": grpo_adv.advantages.tolist(),
       "informative": grpo_adv.informative_groups.tolist(),
       "rloo_first_group": rloo_adv[0].tolist(),
       "all_finite": bool(torch.isfinite(grpo_adv.advantages).all())})

{'grpo_advantages': [[-0.9998000264167786, -0.9998000264167786, 0.9998000264167786, 0.9998000264167786], [0.0, 0.0, 0.0, 0.0]], 'informative': [True, False], 'rloo_first_group': [-0.6666666865348816, -0.6666666865348816, 0.6666666269302368, 0.6666666269302368], 'all_finite': True}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** epsilon만 분모에 더하면 유한값은 만들지만 거의 같은 reward를 과장할 수 있습니다. 명시적 zero-variance mask가 sample budget 낭비를 관찰하게 합니다. RLOO는 다른 baseline 대안입니다.

**흔한 함정:** 서로 다른 prompt의 reward를 한 batch에서 함께 normalize하면 난이도 차이가 credit으로 섞입니다. group axis와 prompt ID를 보존해야 합니다. 회귀 test: `test_group_relative_advantages_and_zero_variance_group`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert grpo_adv.informative_groups.tolist() == [True, False]
assert torch.equal(grpo_adv.advantages[1], torch.zeros(4))
print("checks=passed")

checks=passed


**회상 문제:** critic을 없애면 줄어드는 비용과 새로 커지는 sample 의존성은 각각 무엇인가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 첫 group은 약 ±1 advantage를 만들었고, 상수 reward인 두 번째 group은 정확히 0이며 `informative=False`였습니다.
- 실제 확인: `test_group_relative_advantages_and_zero_variance_group`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L13에서 zero-variance group, 길이 편향, clipping 문제를 DAPO·Dr.GRPO·GSPO 변형으로 분해합니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

[상세 구현 문서](../../docs/algorithms/grpo-family.md) · [강좌 지도](../../docs/course-map.md)

## Sources

- `deepseekmath-grpo-2024` — `docs/sources.yml`
- `rloo-2024` — `docs/sources.yml`
- `repo-deepseek-math` — `docs/sources.yml`